In [1]:
import numpy as np
import pandas as pd

# Question 2

In [2]:
data = pd.read_csv('metabric_survival_homework.csv')

## Section a

In [3]:
print(f"The number of observations in the dataset is: {data.shape[0]}")
print(f"The number of features in the dataset is: {len(set(data.columns) - {'SAMPLE_ID', 'PATIENT_ID', 'time', 'status'})}")  # Exclude ids, 'time' and 'status' columns

censored_pct = (1 - data['status'].mean()) * 100
print(f"Percentage censored: {censored_pct:.1f}%")

print(f"The range of the 'time' column is: {data['time'].min()} to {data['time'].max()} (in months)")

The number of observations in the dataset is: 1980
The number of features in the dataset is: 5006
Percentage censored: 42.3%
The range of the 'time' column is: 0.01 to 355.2 (in months)


## Section b

In [4]:
# precent of missing values in each column
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

missing_valus_cols = missing_values[missing_values > 0].index

TUMOR_STAGE                      514
LYMPH_NODES_EXAMINED_POSITIVE     76
TUMOR_SIZE                        26
POFUT1                             3
CMIP                               2
CTXN1                              2
SLC25A19                           2
BAMBI                              1
IDO1                               1
MRPL24                             1
dtype: int64


In [5]:
# only TUMOR_STAGE is categorical, the rest are numerical
categorical_covariates = ['TUMOR_STAGE']
continuous_covariates = list(set(missing_valus_cols) - set(categorical_covariates))

for col in continuous_covariates:
    data[col] = data[col].fillna(data[col].median())

for col in categorical_covariates:
    mode_value = data[col].mode(dropna=True)[0]
    data[col] = data[col].fillna(mode_value)

## Section c

In [6]:
from lifelines import CoxPHFitter
from joblib import Parallel, delayed
from tqdm import tqdm

cols_surv = list(set(data.columns) - {
    'SAMPLE_ID', 'PATIENT_ID', 'time', 'status', 'AGE_AT_DIAGNOSIS',
    'TUMOR_STAGE', 'TUMOR_SIZE', 'LYMPH_NODES_EXAMINED_POSITIVE', 'ER_STATUS'
})

def fit_one(c, data):
    tmp = data[[c, "time", "status"]].copy()
    cph = CoxPHFitter(penalizer=0.0)
    try:
        cph.fit(tmp, duration_col="time", event_col="status")
        return (c, cph.log_likelihood_)
    except Exception as e:
        return (c, None)

results = Parallel(n_jobs=-1)(
    delayed(fit_one)(c, data) for c in tqdm(cols_surv)
)

utilities = [r for r in results if r[1] is not None]

100%|██████████| 5001/5001 [08:06<00:00, 10.28it/s]


In [7]:
# df of covariates and their utilities
utilities_df = pd.DataFrame(utilities, columns=['covariate', 'utility']).sort_values(by='utility', ascending=False)
print("Top 100 genes by utility:")
print(utilities_df.head(100))

Top 100 genes by utility:
     covariate      utility
3688    FCER1A -7826.141995
1004    PKMYT1 -7826.756594
2876   COL17A1 -7826.950588
2022    JCHAIN -7827.083405
781      STIP1 -7827.346439
...        ...          ...
4756    GALNT6 -7838.291693
1693      ENC1 -7838.348361
3600       TDG -7838.397337
681       KERA -7838.406015
3119     TOP2A -7838.468737

[100 rows x 2 columns]


## Section d

In [8]:
from sklearn.preprocessing import StandardScaler

# Standardize the 100 selected genes
top_100_genes = utilities_df.head(100)['covariate'].tolist()
scaler = StandardScaler()
data[top_100_genes] = scaler.fit_transform(data[top_100_genes])

clinical_covariates = ['AGE_AT_DIAGNOSIS', 'TUMOR_STAGE', 'TUMOR_SIZE', 'LYMPH_NODES_EXAMINED_POSITIVE', 'ER_STATUS']

In [ ]:

from lifelines.utils import k_fold_cross_validation

def build_model_data(data, genes, clinical_covariates):
    df = data[genes + clinical_covariates + ['time', 'status']].copy()
    df = pd.get_dummies(df, columns=['TUMOR_STAGE', 'ER_STATUS'], drop_first=True)
    return df

def penalty_vector(columns, genes, lam):
    # 1 for gene columns (penalized), 0 for everything else (clinical + dummies)
    return np.array([lam if c in genes else 0.0 for c in columns])

def _cv_score_for_lambda(model_data, covariate_cols, genes, l1_ratio, lam, k):
    pen = penalty_vector(covariate_cols, genes, lam)
    cph = CoxPHFitter(penalizer=pen, l1_ratio=l1_ratio)
    try:
        scores = k_fold_cross_validation(
            cph, model_data, duration_col='time', event_col='status',
            k=k, scoring_method='log_likelihood'
        )
        return lam, np.mean(scores)
    except Exception:
        return lam, -np.inf

def fit_cox_model(data, genes, clinical_covariates, l1_ratio,
                      lambdas=(0.001, 0.01, 0.05, 0.1, 0.2), k=5, n_jobs=-1):
    model_data = build_model_data(data, genes, clinical_covariates)
    covariate_cols = [c for c in model_data.columns if c not in ('time', 'status')]

    results = Parallel(n_jobs=n_jobs)(
        delayed(_cv_score_for_lambda)(model_data, covariate_cols, genes, l1_ratio, lam, k)
        for lam in lambdas
    )

    best_lambda, best_cv_score = max(results, key=lambda x: x[1])

    # refit on full data with the CV-selected lambda
    pen = penalty_vector(covariate_cols, genes, best_lambda)
    best_cph = CoxPHFitter(penalizer=pen, l1_ratio=l1_ratio)
    best_cph.fit(model_data, duration_col='time', event_col='status')

    return best_lambda, best_cv_score, best_cph

def get_selected_genes(cph, genes, threshold=1e-4):
    """Genes with non-zero estimated coefficient in a fitted CoxPHFitter model."""
    return set(g for g in genes if abs(cph.params_.get(g, 0)) > threshold)

def report_cox_model_results(lambda_val, cv_score, cph, threshold = 1e-4):
    print(f"Best lambda (5-fold CV): {lambda_val}")
    selected_genes = get_selected_genes(cph, top_100_genes, threshold=threshold)
    print(f"N selected genes: {len(selected_genes)}")
    print("Coefficients (all non-zero covariates):")
    with pd.option_context('display.max_rows', None):
        print(cph.params_[cph.params_.abs() > threshold])

In [10]:
# lasso
lasso_res = fit_cox_model(data, top_100_genes, clinical_covariates, l1_ratio=1)
report_cox_model_results(*lasso_res)

Best lambda (5-fold CV): 0.05
N selected genes: 12
Coefficients (all covariates):
covariate
FCER1A                          -8.340317e-09
PKMYT1                           1.164026e-08
COL17A1                         -3.615490e-09
JCHAIN                          -5.774117e-03
STIP1                            3.670906e-02
GSK3B                            4.105124e-08
STAT5A                          -2.728976e-02
EZR                              1.631587e-08
SPRY2                           -4.538493e-09
RACGAP1                          4.386502e-03
ITM2A                           -9.579296e-09
KIF20A                           1.675311e-08
TROAP                            1.324787e-08
TP63                            -2.258797e-09
OGN                             -9.182591e-09
GLA                             -9.050537e-09
CDCA5                            1.655062e-02
IGF1                            -1.011780e-08
CLDN11                          -4.075466e-09
GRHL2                            1

In [11]:
# elastic net
elastic_net_res = fit_cox_model(data, top_100_genes, clinical_covariates, l1_ratio=0.25)
report_cox_model_results(*elastic_net_res)

Best lambda (5-fold CV): 0.1
N selected genes: 26
Coefficients (all covariates):
covariate
FCER1A                          -7.445992e-09
PKMYT1                           1.273627e-08
COL17A1                         -3.178335e-09
JCHAIN                          -2.255512e-02
STIP1                            2.916857e-02
GSK3B                            1.118712e-02
STAT5A                          -3.860288e-02
EZR                              4.666185e-03
SPRY2                            5.669795e-09
RACGAP1                          9.937254e-03
ITM2A                           -8.384937e-09
KIF20A                           2.550252e-08
TROAP                            1.710903e-08
TP63                             1.809834e-09
OGN                             -1.302460e-08
GLA                             -3.938023e-08
CDCA5                            1.882619e-02
IGF1                            -1.544577e-08
CLDN11                           1.574560e-09
GRHL2                            1.

In [15]:
import matplotlib.pyplot as plt

_, _, lasso_cph = lasso_res
_, _, en_cph = elastic_net_res

lasso_genes = get_selected_genes(lasso_cph, top_100_genes)
en_genes = get_selected_genes(en_cph, top_100_genes)

overlap_genes = lasso_genes & en_genes
lasso_only = lasso_genes - en_genes
en_only = en_genes - lasso_genes
union_genes = lasso_genes | en_genes

jaccard = len(overlap_genes) / len(union_genes) if union_genes else np.nan

print("=== Sparsity comparison ===")
print(f"LASSO selected genes:       {len(lasso_genes)} / {len(top_100_genes)}")
print(f"Elastic-net selected genes: {len(en_genes)} / {len(top_100_genes)}")
print(f"Genes in both models:       {len(overlap_genes)}")
print(f"Genes only in LASSO:        {len(lasso_only)}")
print(f"Genes only in elastic-net:  {len(en_only)}")
print(f"Jaccard similarity:         {jaccard:.3f}")

=== Sparsity comparison ===
LASSO selected genes:       12 / 100
Elastic-net selected genes: 26 / 100
Genes in both models:       12
Genes only in LASSO:        0
Genes only in elastic-net:  14
Jaccard similarity:         0.462


LASSO selected 12 genes while elastic net retained 26, with every gene selected by LASSO also selected by elastic net, meaning the 14 additional elastic-net genes are ones whose coefficients LASSO shrank fully to zero.